# Multimodal Emotion Recognition — Video Inference Demo

Pipeline: **WavLM-Large** (audio) + **GTE finetuned** (text) + **EmotiEffNet-B0** (face) → **QualityAwareGatedFusion**

**Emotions:** Admiration · Amusement · Determination · Empathic Pain · Excitement · Joy

> Run cells top-to-bottom. Each modality section frees GPU memory before the next loads.  
> Edit only **Cell 1 (Config)** to set your video path.

In [ ]:
from pathlib import Path
import torch

# ── User config — edit here ———————————————————————————————————————————
VIDEO_PATH   = Path("/path/to/your/video.mp4")   # ← CHANGE THIS
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
WHISPER_LANG = None   # None = auto-detect; or e.g. 'en', 'ru'

# ── Checkpoint paths (same as training notebooks) ———————————————————————————
DATA_DIR = Path("/home/danila/networks/data")
RUNS_DIR = DATA_DIR / "runs"

AUDIO_ENCODER_DIR = RUNS_DIR / "audio_wavlm_end2end_fps5_v1" / "finetuned_encoder"
AUDIO_HEAD_PT     = RUNS_DIR / "audio_wavlm_end2end_fps5_v1" / "finetuned_head" / "head.pt"
TEXT_ENCODER_DIR  = RUNS_DIR / "finetune_mgte_text_only_v1"  / "best_encoder_for_embeddings"
TEXT_HEAD_PT      = RUNS_DIR / "finetune_mgte_text_only_v1"  / "best_by_val_pearson.pt"
FACE_HEAD_PT      = RUNS_DIR / "abaw8_emi_face_repro"          / "best_by_corr.pt"
FUSION_PT         = RUNS_DIR / "multimodal_fusion_emi_face_v1" / "quality_gated" / "best_fusion.pt"
CALIB_PATH        = RUNS_DIR / "multimodal_fusion_emi_face_v1" / "calibration_params.npz"
NORM_PATH         = RUNS_DIR / "multimodal_fusion_emi_face_v1" / "target_norm.npz"

# ── Pipeline constants (must match training) ——————————————————————————————
TARGET_SR     = 16_000
AUDIO_MAX_SEC = 20.0
TARGET_FPS    = 5
HOP_SEC       = 1.0 / TARGET_FPS   # 0.2 s per bin
NUM_BINS      = int(AUDIO_MAX_SEC / HOP_SEC)  # 100

WHISPER_MODEL  = "large-v3"
TEXT_MAX_LEN   = 128

FACE_MIN_PROB  = 0.90
FACE_MIN_VALID = 5
FACE_MARGIN    = 0.20
FACE_ENC_BATCH = 64
MTCNN_BATCH    = 16

EMOTIONS = ["Admiration", "Amusement", "Determination", "Empathic Pain", "Excitement", "Joy"]
E = len(EMOTIONS)

print(f"Device : {DEVICE}")
print(f"Video  : {VIDEO_PATH}")


In [ ]:
import gc
import subprocess
import tempfile

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image


# ── Audio model classes (exact copies from training notebooks) ──────────────

class TCNBlock(nn.Module):
    def __init__(self, d, kernel, dilation, dropout):
        super().__init__()
        padding = (kernel - 1) * dilation // 2
        self.conv1 = nn.Conv1d(d, d, kernel_size=kernel, dilation=dilation, padding=padding)
        self.conv2 = nn.Conv1d(d, d, kernel_size=kernel, dilation=dilation, padding=padding)
        self.norm1 = nn.GroupNorm(1, d)
        self.norm2 = nn.GroupNorm(1, d)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x):
        res = x
        x = self.drop(F.gelu(self.norm1(self.conv1(x))))
        x = self.drop(F.gelu(self.norm2(self.conv2(x))))
        return x + res


class TCNEncoder(nn.Module):
    def __init__(self, d, layers, kernel, dropout):
        super().__init__()
        self.blocks = nn.ModuleList([TCNBlock(d, kernel, 2**i, dropout) for i in range(layers)])

    def forward(self, x):   # [B, T, d]
        x = x.transpose(1, 2)
        for b in self.blocks:
            x = b(x)
        return x.transpose(1, 2)


class AttentiveStatsPooling(nn.Module):
    def __init__(self, d, attn_hidden, dropout, temp=1.5):
        super().__init__()
        self.temp = temp
        self.attn = nn.Sequential(
            nn.Linear(d, attn_hidden), nn.Tanh(),
            nn.Dropout(dropout), nn.Linear(attn_hidden, 1),
        )

    def forward(self, x, mask):   # x: [B, T, d], mask: [B, T] bool
        logits = self.attn(x).squeeze(-1) / self.temp
        logits = logits.masked_fill(~mask, -1e4)
        w = torch.softmax(logits, dim=1)
        w = w * mask.float()
        w = w / (w.sum(dim=1, keepdim=True) + 1e-6)
        w = w.unsqueeze(-1)
        mu  = (w * x).sum(dim=1)
        var = (w * (x - mu.unsqueeze(1)).pow(2)).sum(dim=1)
        std = torch.sqrt(var + 1e-6)
        return torch.cat([mu, std], dim=-1)


class AudioWavLMModel(nn.Module):
    # TCN + Attentive Stats Pooling head on pre-binned WavLM embeddings
    def __init__(self, h_in, d_model=192, tcn_layers=6, tcn_kernel=3, dropout=0.3, attn_hidden=128):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(h_in, d_model), nn.LayerNorm(d_model), nn.Dropout(dropout),
        )
        self.enc  = TCNEncoder(d_model, tcn_layers, tcn_kernel, dropout)
        self.pool = AttentiveStatsPooling(d_model, attn_hidden, dropout, temp=1.5)
        self.head = nn.Sequential(
            nn.Linear(2*d_model, 2*d_model), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(2*d_model, E),
        )

    def forward(self, x, mask):
        x = self.proj(x)
        x = self.enc(x)
        z = self.pool(x, mask)
        return self.head(z)


class EmiFaceHead(nn.Module):
    # Linear -> Sigmoid (matches upstream Keras Dense(6, activation='sigmoid'))
    def __init__(self, in_dim, num_targets):
        super().__init__()
        self.fc = nn.Linear(in_dim, num_targets)
    def forward(self, x):
        return torch.sigmoid(self.fc(x))


class QualityAwareGatedFusion(nn.Module):
    def __init__(self, hidden=32):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(E + 2, hidden), nn.GELU(), nn.Linear(hidden, E),
        )
        self.bias = nn.Parameter(torch.zeros(E))

    def forward(self, P, avail, quality):
        # P: [B, 3, E], avail: [B, 3] bool, quality: [B, 3] float
        feat   = torch.cat([P, quality.unsqueeze(-1), avail.unsqueeze(-1).float()], dim=-1)
        logits = self.mlp(feat)
        mask   = avail.unsqueeze(-1).expand_as(logits)
        logits = logits.masked_fill(~mask, -1e4)
        w      = torch.softmax(logits, dim=1)
        w      = w * mask.float()
        w      = w / (w.sum(dim=1, keepdim=True) + 1e-6)
        fused  = (w * P).sum(dim=1) + self.bias
        return fused, w


# ── Helper functions ────────────────────────────────────────────────────────

def hidden_to_fps_bins(hidden, input_lengths, output_lengths, sr, hop_sec, num_bins):
    # Map WavLM hidden [B, Lmax, H] to fixed-size 5-fps bins [B, num_bins, H]
    B, Lmax, H = hidden.shape
    device = hidden.device
    dur    = input_lengths.float() / float(sr)
    x_out, m_out = [], []
    for b in range(B):
        Lb = int(output_lengths[b].item())
        if Lb <= 0:
            x_out.append(torch.zeros((num_bins, H), device=device, dtype=hidden.dtype))
            m_out.append(torch.zeros((num_bins,),  device=device, dtype=torch.bool))
            continue
        hb     = hidden[b, :Lb, :]
        stride = dur[b] / float(Lb)
        t      = torch.arange(Lb, device=device, dtype=torch.float32) * stride
        idx    = torch.floor(t / float(hop_sec)).long()
        valid  = (idx >= 0) & (idx < num_bins)
        sums   = torch.zeros((num_bins, H), device=device, dtype=hidden.dtype)
        cnts   = torch.zeros((num_bins,),  device=device, dtype=torch.int32)
        if valid.any():
            ii = idx[valid]
            sums.index_add_(0, ii, hb[valid])
            cnts.index_add_(0, ii, torch.ones_like(ii, dtype=torch.int32))
        xb = sums / cnts.clamp_min(1).unsqueeze(1).to(sums.dtype)
        mb = cnts > 0
        x_out.append(xb); m_out.append(mb)
    return torch.stack(x_out), torch.stack(m_out)


def pool_face_emb(emb: np.ndarray, valid: np.ndarray) -> np.ndarray:
    # mean + std + min + max over valid frames -> [4*D]
    x      = emb[valid].astype(np.float32) if valid.any() else emb.astype(np.float32)
    mean_f = x.mean(0)
    std_f  = x.std(0)  if x.shape[0] > 1 else np.zeros_like(mean_f)
    min_f  = x.min(0)
    max_f  = x.max(0)
    return np.concatenate([mean_f, std_f, min_f, max_f]).astype(np.float32)


def crop_face(img_rgb: np.ndarray, box: np.ndarray, margin: float = 0.20) -> np.ndarray:
    H, W = img_rgb.shape[:2]
    x1, y1, x2, y2 = box
    pw = (x2 - x1) * margin;  ph = (y2 - y1) * margin
    x1 = max(0, int(x1 - pw)); y1 = max(0, int(y1 - ph))
    x2 = min(W, int(x2 + pw)); y2 = min(H, int(y2 + ph))
    return img_rgb[y1:y2, x1:x2]


def free_gpu(*objs):
    for o in objs:
        del o
    gc.collect()
    torch.cuda.empty_cache()


print("All classes and helpers defined.")


---
## Section 1 — Audio

1. Extract 16 kHz mono WAV from video with **ffmpeg**
2. Compute **WavLM-Large** hidden states → bin to 5 fps → **free WavLM** (~1.3 B params)
3. Apply TCN + Attentive Stats Pooling head → 6-D emotion prediction


In [ ]:
result = subprocess.run(
    [
        "ffmpeg",
        "-hide_banner",
        "-loglevel", "error",
        "-i", str(VIDEO_PATH),
        "-vn",
        "-ac", "1",
        "-ar", str(TARGET_SR),
        "-t", str(AUDIO_MAX_SEC),
        "-f", "f32le",
        "pipe:1",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

if result.returncode != 0:
    raise RuntimeError(f"ffmpeg failed:\n{result.stderr.decode(errors='replace')}")

wav_np = np.frombuffer(result.stdout, dtype=np.float32).copy()

if wav_np.size == 0:
    raise RuntimeError("ffmpeg returned empty audio. The video may have no audio stream.")

wav = torch.from_numpy(wav_np)  # shape: [T], dtype: float32

print(f"Audio: {wav.shape[0] / TARGET_SR:.1f}s  ({wav.shape[0]} samples)")

In [ ]:
from transformers import WavLMModel, AutoFeatureExtractor

print("Loading WavLM encoder ...")
wavlm    = WavLMModel.from_pretrained(str(AUDIO_ENCODER_DIR)).to(DEVICE).eval()
feat_ext = AutoFeatureExtractor.from_pretrained(str(AUDIO_ENCODER_DIR))

fe = feat_ext(wav.numpy(), sampling_rate=TARGET_SR, return_tensors='pt')
fe = {k: v.to(DEVICE) for k, v in fe.items()}

with torch.no_grad():
    attn_mask = fe.get(
        "attention_mask",
        torch.ones(1, fe['input_values'].shape[1], device=DEVICE, dtype=torch.long),
    )
    input_lengths  = attn_mask.sum(-1)
    output_lengths = wavlm._get_feat_extract_output_lengths(input_lengths)
    out    = wavlm(**fe)
    hidden = out.last_hidden_state   # [1, Lmax, 1024]

    audio_bins, audio_mask = hidden_to_fps_bins(
        hidden, input_lengths, output_lengths, TARGET_SR, HOP_SEC, NUM_BINS,
    )

audio_bins    = audio_bins.cpu().float()
audio_mask    = audio_mask.cpu()
audio_quality = float(audio_mask[0].float().mean())

free_gpu(wavlm, out, hidden, fe)
print(f"Bins shape : {audio_bins.shape}")
print(f"Valid bins : {audio_mask[0].sum().item()} / {NUM_BINS}")
print(f"Quality    : {audio_quality:.3f}")


In [ ]:
pkg      = torch.load(AUDIO_HEAD_PT, map_location='cpu')
h_in     = int(pkg['h_in'])
num_bins = int(pkg['num_bins'])

audio_head = AudioWavLMModel(h_in=h_in).to(DEVICE).eval()
audio_head.load_state_dict(pkg['head_state'], strict=True)

# Align temporal dimension to training num_bins
T = audio_bins.shape[1]
if T < num_bins:
    pad = num_bins - T
    audio_bins = torch.cat([audio_bins, torch.zeros(1, pad, h_in)], dim=1)
    audio_mask = torch.cat([audio_mask, torch.zeros(1, pad, dtype=torch.bool)], dim=1)
elif T > num_bins:
    audio_bins = audio_bins[:, :num_bins]
    audio_mask = audio_mask[:, :num_bins]

with torch.no_grad():
    audio_pred = audio_head(audio_bins.to(DEVICE), audio_mask.to(DEVICE))
    audio_pred = audio_pred.cpu().numpy()[0]   # [E]

audio_avail = bool(audio_mask[0].any().item())
free_gpu(audio_head)

print(f"Audio avail: {audio_avail}")
for em, sc in zip(EMOTIONS, audio_pred):
    print(f"  {em:<16}: {sc:+.4f}")


---
## Section 2 — Text

1. Transcribe speech with **Whisper large-v3** → **free Whisper**
2. Encode transcript with **GTE finetuned encoder** → CLS embedding → **free GTE**
3. Apply MLP head (weights extracted from checkpoint) → 6-D emotion prediction


In [ ]:
from faster_whisper import WhisperModel as FasterWhisper

compute_type = "float16" if DEVICE == "cuda" else "int8"
print(f"Loading Whisper {WHISPER_MODEL} ({compute_type}) ...")
whisper = FasterWhisper(
    WHISPER_MODEL,
    device="cuda" if DEVICE == "cuda" else "cpu",
    compute_type=compute_type,
)

segments, info = whisper.transcribe(str(VIDEO_PATH), language=WHISPER_LANG)
transcript = ' '.join(s.text.strip() for s in segments).strip() or '[NO SPEECH]'

lang_detected = getattr(info, 'language', '?')
free_gpu(whisper)

print(f"Language : {lang_detected}")
print(f"Transcript ({len(transcript)} chars):")
print(" ", transcript[:400])


In [ ]:
from transformers import AutoModel, AutoTokenizer

print("Loading GTE encoder ...")
gte = AutoModel.from_pretrained(str(TEXT_ENCODER_DIR), trust_remote_code=True).to(DEVICE).eval()
tok = AutoTokenizer.from_pretrained(str(TEXT_ENCODER_DIR), use_fast=True)

inputs = tok(
    [transcript], padding=True, truncation=True,
    max_length=TEXT_MAX_LEN, return_tensors='pt',
)
inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

with torch.no_grad():
    out     = gte(**inputs)
    cls_emb = out.last_hidden_state[:, 0, :].float()   # [1, 768]
    text_quality = float(torch.tanh(cls_emb.norm() / 10.0))
    cls_cpu = cls_emb.cpu()

free_gpu(gte, out, inputs)

# Extract head weights from mGTERegressor checkpoint
ckpt_t  = torch.load(TEXT_HEAD_PT, map_location='cpu')
state   = ckpt_t.get('model_state', ckpt_t)
head_sd = {k.replace('head.', ''): v for k, v in state.items() if k.startswith('head.')}
assert head_sd, "No 'head.*' keys found in text checkpoint!"

D_TEXT    = int(cls_cpu.shape[-1])
text_head = nn.Sequential(
    nn.Dropout(0.2), nn.Linear(D_TEXT, 512), nn.GELU(),
    nn.Dropout(0.2), nn.Linear(512, E),
).eval()
text_head.load_state_dict(head_sd, strict=True)

with torch.no_grad():
    text_pred = text_head(cls_cpu).numpy()[0]   # [E]

text_avail = transcript != '[NO SPEECH]'
free_gpu(text_head, cls_cpu)

print(f"Text quality : {text_quality:.3f}  |  avail: {text_avail}")
for em, sc in zip(EMOTIONS, text_pred):
    print(f"  {em:<16}: {sc:+.4f}")


---
## Section 3 — Face

1. Extract frames at ≈5 fps with **OpenCV**
2. Detect faces with **MTCNN** → **free MTCNN**
3. Encode face crops with **EmotiEffNet-B0** (`enet_b0_8_va_mtl`) → **free encoder**
4. Pool [mean, std, min, max] → 5120-D → Linear+Sigmoid head → 6-D prediction


In [ ]:
from facenet_pytorch import MTCNN
from emotiefflib.facial_analysis import EmotiEffLibRecognizer

# Extract frames at ~TARGET_FPS using OpenCV
cap     = cv2.VideoCapture(str(VIDEO_PATH))
fps_src = cap.get(cv2.CAP_PROP_FPS) or 25.0
stride  = max(1, round(fps_src / TARGET_FPS))
frames  = []
idx     = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    if idx % stride == 0:
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    idx += 1
cap.release()
print(f"Extracted {len(frames)} frames  (src_fps={fps_src:.1f}, stride={stride})")

# Detect faces with MTCNN
mtcnn = MTCNN(keep_all=True, device=DEVICE,
              thresholds=[0.6, 0.7, 0.7], min_face_size=40, post_process=False)

def _largest(boxes):
    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    return int(np.argmax(areas))

face_imgs  = []
face_probs = []

for b0 in range(0, len(frames), MTCNN_BATCH):
    batch     = frames[b0:b0 + MTCNN_BATCH]
    pil_batch = [Image.fromarray(f) for f in batch]
    boxes_l, probs_l = mtcnn.detect(pil_batch)
    for frame_np, boxes, probs in zip(batch, boxes_l, probs_l):
        if boxes is None or probs is None:
            continue
        keep = probs >= FACE_MIN_PROB
        if not keep.any():
            continue
        bk = boxes[keep]; pk = probs[keep]
        k  = _largest(bk)
        crop = crop_face(frame_np, bk[k], FACE_MARGIN)
        if crop.size == 0:
            continue
        face_imgs.append(crop)
        face_probs.append(float(pk[k]))

del mtcnn
torch.cuda.empty_cache(); gc.collect()
print(f"Faces detected: {len(face_imgs)}")

# Encode with EmotiEffNet-B0
encoder = EmotiEffLibRecognizer(engine='torch', model_name='enet_b0_8_va_mtl', device=DEVICE)

emb_chunks = []
for b0 in range(0, len(face_imgs), FACE_ENC_BATCH):
    feats = encoder.extract_features(face_imgs[b0:b0 + FACE_ENC_BATCH])
    emb_chunks.append(feats)

del encoder
torch.cuda.empty_cache(); gc.collect()

if emb_chunks:
    face_emb_arr  = np.concatenate(emb_chunks, axis=0).astype(np.float32)
    face_valid_arr = np.ones(len(face_emb_arr), dtype=bool)
    face_pool      = pool_face_emb(face_emb_arr, face_valid_arr)
    face_mean_prob = float(np.mean(face_probs))
    print(f"Embedding shape : {face_emb_arr.shape}")
    print(f"Mean det prob   : {face_mean_prob:.3f}")
else:
    face_pool      = None
    face_mean_prob = 0.0
    print("No faces found — face modality will be marked unavailable.")


In [ ]:
emi_ckpt = torch.load(FACE_HEAD_PT, map_location='cpu')
in_dim   = int(emi_ckpt['in_dim'])

face_head = EmiFaceHead(in_dim, E).eval()
face_head.load_state_dict(emi_ckpt['model_state'], strict=True)

if face_pool is not None and len(face_imgs) >= FACE_MIN_VALID:
    xt = torch.from_numpy(face_pool).unsqueeze(0)
    with torch.no_grad():
        face_pred = face_head(xt).numpy()[0]   # [E]
    face_avail   = True
    face_quality = face_mean_prob
else:
    face_pred    = np.zeros(E, dtype=np.float32)
    face_avail   = False
    face_quality = 0.0

free_gpu(face_head)

print(f"Face avail: {face_avail}  |  quality: {face_quality:.3f}")
for em, sc in zip(EMOTIONS, face_pred):
    print(f"  {em:<16}: {sc:+.4f}")


---
## Section 4 — Fusion & Visualization

Combines the three 6-D predictions with **QualityAwareGatedFusion**.  
Applies optional calibration (from training data) and target denormalization, then plots results.


In [ ]:
# Stack per-modality inputs
preds  = np.stack([audio_pred, text_pred, face_pred], axis=0)   # [3, E]
avails = np.array([audio_avail, text_avail, face_avail], dtype=bool)
quals  = np.array([audio_quality, text_quality, face_quality], dtype=np.float32)

print("Per-modality predictions (pre-calibration):")
for name, pred, av in zip(['Audio', 'Text ', 'Face '], preds, avails):
    tag  = '✓' if av else '✗'
    vals = np.array2string(pred, precision=3, suppress_small=True)
    print(f"  {tag} {name}: {vals}")

# Optional calibration
if CALIB_PATH.exists():
    cal = np.load(CALIB_PATH, allow_pickle=False)
    preds[0] = preds[0] * cal['a_audio'] + cal['b_audio']
    preds[1] = preds[1] * cal['a_text']  + cal['b_text']
    preds[2] = preds[2] * cal['a_video'] + cal['b_video']
    print("\nCalibration applied.")
else:
    print("\nCalibration file not found — skipping.")

# Fusion
if not FUSION_PT.exists():
    raise FileNotFoundError(
        f"Fusion checkpoint not found:\n  {FUSION_PT}\n"
        "Run additional_experements/multimodal_fusion_emi_face_v1.ipynb first."
    )

fusion = QualityAwareGatedFusion(hidden=32).eval()
ckpt   = torch.load(FUSION_PT, map_location='cpu')
fusion.load_state_dict(ckpt['model_state'], strict=True)

P_t = torch.from_numpy(preds).unsqueeze(0).float()     # [1, 3, E]
A_t = torch.from_numpy(avails).unsqueeze(0)            # [1, 3]
Q_t = torch.from_numpy(quals).unsqueeze(0).float()     # [1, 3]

with torch.no_grad():
    fused_norm, weights = fusion(P_t, A_t, Q_t)
fused_norm = fused_norm.numpy()[0]   # [E]
weights    = weights.numpy()[0]      # [3, E]

# Optional denormalization
if NORM_PATH.exists():
    norm  = np.load(NORM_PATH, allow_pickle=False)
    fused = fused_norm * norm['y_std'] + norm['y_mean']
    print("Target denormalization applied.")
else:
    fused = fused_norm
    print("target_norm.npz not found — showing z-score scale output.")

print("\nFinal emotion vector:")
for em, sc in zip(EMOTIONS, fused):
    print(f"  {em:<16}: {sc:.4f}")

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"Emotion Recognition — {VIDEO_PATH.name}", fontsize=13, fontweight="bold")

# Left: emotion bar chart
cmap   = plt.cm.RdYlGn
vnorm  = plt.Normalize(fused.min(), fused.max())
colors = [cmap(vnorm(v)) for v in fused]
bars   = ax1.barh(EMOTIONS, fused, color=colors, edgecolor='grey', linewidth=0.5)
ax1.set_xlabel("Predicted Score", fontsize=11)
ax1.set_title("Emotion Vector", fontsize=12)
span   = max(abs(fused).max() * 1.18, 0.05)
ax1.set_xlim(-span if fused.min() < 0 else 0, span)
ax1.axvline(0, color='black', linewidth=0.8, linestyle='--')
for bar, val in zip(bars, fused):
    xpos = bar.get_width() + span * 0.02
    ax1.text(xpos, bar.get_y() + bar.get_height() / 2, f'{val:.3f}',
             va='center', fontsize=9)

# Right: modality gate weights (mean across emotions)
mod_names = ['Audio', 'Text', 'Face']
mod_cols  = ['#4CAF50', '#2196F3', '#FF9800']
mean_w    = weights.mean(axis=1)   # [3]
ax2.bar(mod_names, mean_w, color=mod_cols, edgecolor='grey', linewidth=0.5)
ax2.set_ylabel("Mean Gate Weight", fontsize=11)
ax2.set_title("Modality Contributions", fontsize=12)
ax2.set_ylim(0, max(1.0, mean_w.max() * 1.35))
for i, (w, av) in enumerate(zip(mean_w, avails)):
    label = f'{w:.3f}' + ('' if av else '\n(unavail.)')
    ax2.text(i, w + 0.01, label, ha='center', fontsize=10,
             color='black' if av else 'gray')

plt.tight_layout()
plt.savefig("emotion_prediction.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to emotion_prediction.png")
